In [ ]:
#I. Chargement des données

import seaborn as sns
import pandas as pd
titanic = pd.read_csv('/kaggle/input/titanic/train.csv')
print(titanic)


In [ ]:
#II. Compréhension des données
# TODO: Imprimer les premières lignes de l'ensemble de données
titanic.head()
# TODO: Afficher un résumé statistique de l'ensemble de données
titanic.describe()



In [ ]:
#III. Prétraitement des données 
from sklearn.preprocessing import StandardScaler
# TODO : # Remplissez les valeurs manquantes pour 'age'
titanic['Age']=titanic['Age'].fillna(titanic['Age'].mean())
scaler=StandardScaler()
titanic['age_scaled']=scaler.fit_transform(titanic[['Age']])
# Encoder la colonne 'sex'
titanic['Sex'] = titanic['Sex'].replace({'male': 0, 'female': 1})
# Diviser les données
from sklearn.model_selection import train_test_split
X = titanic[['Pclass', 'Sex', 'Age']]
y = titanic['Survived']
# TODO: Divisez les données en ensembles de formation et de validation
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.2,random_state=42)
# TODO: Afficher X et y
print('X(features) :')
print(X.head())
print('y(cible) :')
print(y.head())

In [ ]:
# IV. Construction de modèles
from keras.models import Sequential
from keras.layers import Dense

num_features = X_train.shape[1]

# TODO : Ajouter une couche d'entrée et cachée avec 50 et 20 neurones respectivement
model = Sequential()  # CORRECTION: = au lieu de -
model.add(Dense(units=50, activation='relu', input_dim=num_features))  
model.add(Dense(20, activation='relu'))  

# TODO : Ajouter une couche de sortie adaptée à la classification binaire
model.add(Dense(1, activation='sigmoid'))  # CORRECTION: 1 neurone avec sigmoid pour classification binaire

# TODO : Compiler le modèle avec la perte, l'optimiseur et les métriques appropriés
model.compile(
    loss='binary_crossentropy',  # Pour la classification binaire
    optimizer='adam',            # Optimiseur populaire
    metrics=['accuracy']         # CORRECTION: liste ['accuracy'] au lieu de string
)

# Vérifier l'architecture
model.summary()

In [ ]:
# V. Formation et évaluation
# TODO : Entraîner le modèle sur les données d'entraînement et valider sur les données de validation avec 50 epochs
history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val),
    verbose=1  # CORRECTION: ajouter verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()


In [ ]:
# TODO : Utilisez le modèle entraîné pour faire des prédictions sur l'ensemble de validation
predictions= model.predict(X_val)

# Afficher les premières prédictions (probabilités)
print("Prédictions brutes (probabilités) :")
print(predictions[:10])

# Convertir les probabilités en classes (0 ou 1)
predictions_classes = (predictions> 0.5).astype("int32")

print("\nPrédictions converties en classes :")
print(predictions_classes[:10].flatten())

print("\nVraies valeurs (y_val) :")
print(y_val[:10].values)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# TODO : Calculer diverses métriques d'évaluation et l'afficher
accuracy = accuracy_score(y_val, predictions_classes)
precision = precision_score(y_val, predictions_classes)
recall = recall_score(y_val,predictions_classes)
f1 = f1_score(y_val, predictions_classes)

print("=== MÉTRIQUES D'ÉVALUATION ===")
print(f"Accuracy (Exactitude): {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Rappel): {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Tracez la matrice de confusion
confusion = confusion_matrix(y_val, predictions_classes)
plt.figure(figsize=(6, 6))
plt.imshow(confusion, interpolation='nearest', cmap=plt.cm.inferno)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ["Not Survived", "Survived"])
plt.yticks([0, 1], ["Not Survived", "Survived"])
plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
rows, cols = confusion.shape
for i in range(rows):
    for j in range(cols):
        plt.text(j, i, confusion[i, j],
                 horizontalalignment='center',
                 color='white' if confusion[i, j] > (0.5*confusion.max()) else 'black')
plt.show()

In [ ]:
#Student Performance prediction
# I. Chargement des données

import pandas as pd
import numpy as np

np.random.seed(42)
n = 500
df = pd.DataFrame({
    'study_hours': np.random.uniform(1, 20, n),
    'attendance': np.random.uniform(50, 100, n),
    'previous_grade': np.random.uniform(50, 100, n),
    'gender': np.random.choice(['male', 'female'], n),
    'parent_education': np.random.choice(['high school', 'bachelor', 'master'], n),
    'extracurricular': np.random.choice(['yes','no'], n),
})

df['final_score'] = (
    0.4 * df['study_hours'] + 
    0.3 * df['attendance'] + 
    0.2 * df['previous_grade'] + 
    np.random.normal(0,5,n)
)
df['final_score'] = df['final_score'].clip(0,100)
df.head()

In [ ]:
# II. Prétraitement des données 
# TODO: Encoder la colonne 'gender'
df['gender'] = df['gender'].map({'male': 0, 'female': 1})

# TODO: Appliquer le one-hot encoder sur les features (parent_education, extracurricular)
df = pd.get_dummies(df, columns=['parent_education', 'extracurricular'], drop_first=True)

from sklearn.preprocessing import StandardScaler
numeric_features = ['study_hours','attendance','previous_grade']

# TODO: Appliquer le standard scaling sur les features numériques
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

from sklearn.model_selection import train_test_split
X = df.drop('final_score', axis=1)
y = df['final_score']

# TODO: Divisez les données en ensembles de formation et de validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Prétraitement terminé !")
print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
print(f"Colonnes: {X.columns.tolist()}")

In [ ]:
#III. Construction de modèles
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)

from keras.models import Sequential
from keras.layers import Dense

# TODO : Ajouter une couche d'entrée et cachée avec 50 et 20 neurones respectivement
model = Sequential()
model.add(Dense(50, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(20, activation='relu'))

# TODO : Ajouter une couche de sortie adaptée à la regression
model.add(Dense(1, activation='linear'))


In [ ]:
# IV. Formation et évaluation 
# TODO : Compiler le modèle avec la perte, l'optimiseur et les métriques appropriés
model.compile(
    loss='mse',           # Mean Squared Error pour la régression
    optimizer='adam',     # Optimiseur adaptatif
    metrics=['mae']       # Mean Absolute Error comme métrique
)

# TODO : Entraîner le modèle sur les données d'entraînement et valider sur les données de validation avec 50 epochs
history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val),
    verbose=1,
    batch_size=32
)

In [ ]:
#V. Prédiction 
# TODO : Utilisez le modèle entraîné pour faire des prédictions sur l'ensemble de validation
predictions = model.predict(X_val)

# Afficher les premières prédictions
print("Prédictions sur l'ensemble de validation :")
print(predictions[:10].flatten())

print("\nVraies valeurs (y_val) :")
print(y_val[:10])

In [ ]:
# VI. Metriques d'évaluation 
from sklearn.metrics import mean_absolute_error, r2_score

# TODO  Calculer l'erreur absolue moyenne (MAE) et la valeur R au carré.
from sklearn.metrics import mean_absolute_error, r2_score

# TODO : Calculer l'erreur absolue moyenne (MAE) et la valeur R au carré
mae = mean_absolute_error(y_val, predictions)
r2 = r2_score(y_val, predictions)

print("=== MÉTRIQUES DE RÉGRESSION ===")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"R² Score: {r2:.4f}")
